In [1]:
import re
import string

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from tqdm.notebook import tqdm  # Barre de progression pour Jupyter

In [3]:
# dataset1 = dataset mono-langue brut
dataset1 = pd.read_csv("../../data/raw/mono-langue.csv")
df_mono = dataset1.copy()

In [4]:
# =============================================================================
# 1. TÉLÉCHARGEMENT DES RESSOURCES NLTK (une seule fois)
# =============================================================================
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # Requis pour les versions récentes de NLTK
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [5]:
# Fonctions de preprocessing definies localement (sans import externe)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer() # Le lemmatizer sert à ramener les mots à leur forme de base.

def _normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_classic(text: str) -> str:
    text = _normalize_text(text)
    tokens = [
        token for token in word_tokenize(text)
        if token.isalpha() and token not in stop_words
    ]
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmas)

def preprocess_light(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Fonctions preprocess_classic et preprocess_light chargees localement.")

Fonctions preprocess_classic et preprocess_light chargees localement.


## Code du Pipeline 1 — Classique

In [6]:
# =============================================================================
# INITIALISATION DES OUTILS
# =============================================================================
stop_words = set(stopwords.words('english'))  # Mettre 'french' si votre mono est en français
lemmatizer = WordNetLemmatizer()
# =============================================================================
# APPLICATION SUR df_mono
# =============================================================================
print(" Lancement du Pipeline 1 — Classique...")
df_mono['text_clean_classique'] = df_mono['text'].apply(preprocess_classic)
print(" Préprocessing terminé !")

# =============================================================================
# VÉRIFICATION VISUELLE (Before / After)
# =============================================================================
pd.set_option('display.max_colwidth', 80)
display(df_mono[['text', 'text_clean_classique']].head(5))

 Lancement du Pipeline 1 — Classique...
 Préprocessing terminé !


,text,text_clean_classique
0,Great value for money,great value money
1,"I love it, will definitely buy again",love definitely buy
2,Works perfectly and arrived quickly,work perfectly arrived quickly
3,Decent quality for the price,decent quality price
4,"Terrible product, broke after one use",terrible product broke one use


## Code du Pipeline Light

In [7]:
# =============================================================================
# 2. APPLICATION SUR VOS DATASETS
# =============================================================================
print("🧹 Application du nettoyage léger sur df_mono...")
df_mono['text_clean_light'] = df_mono['text'].apply(preprocess_light)

print("✅ Nettoyage terminé !")

🧹 Application du nettoyage léger sur df_mono...
✅ Nettoyage terminé !


In [8]:
# Afficher 5 exemples pour validation visuelle
pd.set_option('display.max_colwidth', 100)
comparison = df_mono[['text', 'text_clean_classique', 'text_clean_light']].head(5)
print(comparison.to_markdown(index=False))

| text                                  | text_clean_classique           | text_clean_light                      |
|:--------------------------------------|:-------------------------------|:--------------------------------------|
| Great value for money                 | great value money              | great value for money                 |
| I love it, will definitely buy again  | love definitely buy            | i love it, will definitely buy again  |
| Works perfectly and arrived quickly   | work perfectly arrived quickly | works perfectly and arrived quickly   |
| Decent quality for the price          | decent quality price           | decent quality for the price          |
| Terrible product, broke after one use | terrible product broke one use | terrible product, broke after one use |


In [10]:
df_mono.to_csv("../../data/processed/mono-langue_clean.csv", index=False)